In [40]:
from langchain_groq import ChatGroq
from config import settings

In [41]:
llm = ChatGroq(
    temperature=0, 
    groq_api_key=settings.GROQ_API, 
    model_name="openai/gpt-oss-120b"
)
response = llm.invoke("The first person to land on moon was ...")
print(response.content)

The first person to set foot on the Moon was **Neil Armstrong**, who did so on July 20 1945 (Apollo 11 mission).


In [42]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://jobs.nike.com/job/R-33460")
page_data = loader.load().pop().page_content
print(page_data)





















Nike Careers












































Skip to main content
Open Virtual Assistant










Home


Career Areas


Total Rewards


Life@Nike


Purpose










Language





Select a Language

  Deutsch  
  English  
  Español (España)  
  Español (América Latina)  
  Français  
  Italiano  
  Nederlands  
  Polski  
  Tiếng Việt  
  Türkçe  
  简体中文  
  繁體中文  
  עִברִית  
  한국어  
  日本語  








Careers


















Close Menu







Careers






Chat






                                Home
                            



                                Career Areas
                            



                                Total Rewards
                            



                                Life@Nike
                            



                                Purpose
                            










Jordan Careers







Converse Careers










Language











Menu



Return to Previous Menu



Select a Lan

In [43]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
        """
        ### SCRAPED TEXT FROM WEBSITE:
        {page_data}
        ### INSTRUCTION:
        The scraped text is from the career's page of a website.
        Your job is to extract the job postings and return them in JSON format containing the 
        following keys: `role`, `experience`, `skills` and `description`.
        Only return the valid JSON.
        ### VALID JSON (NO PREAMBLE):    
        """
)

chain_extract = prompt_extract | llm 
res = chain_extract.invoke(input={'page_data':page_data})
type(res.content)

str

In [44]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
json_res

[{'role': 'Retail Associate, PT - Nike Las Vegas South',
  'experience': '',
  'skills': '',
  'description': ''},
 {'role': 'Visual Merchandising & Styling Lead',
  'experience': '',
  'skills': '',
  'description': ''},
 {'role': 'Retail Associate, FT - Nike Memphis Company Store',
  'experience': '',
  'skills': '',
  'description': ''},
 {'role': 'Retail Associate, SEAS - Nike Barstow',
  'experience': '',
  'skills': '',
  'description': ''},
 {'role': 'Retail Associate, SEAS - Nike East LA Community',
  'experience': '',
  'skills': '',
  'description': ''},
 {'role': 'Retail Associate, SEAS - Nike Clinton',
  'experience': '',
  'skills': '',
  'description': ''},
 {'role': 'Designer II, NSW Print & Pattern Graphic Design',
  'experience': '',
  'skills': '',
  'description': ''},
 {'role': 'Stockroom Retail Assistant (Athlete) - PT 32 HRS - NIKE SUGAR CITY',
  'experience': '',
  'skills': '',
  'description': ''},
 {'role': 'Software Engineer II, IT',
  'experience': '',
  'sk

In [45]:
type(json_res)

list

In [46]:
import pandas as pd

df = pd.read_csv("my_portfolio.csv")
df

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio
5,"Flutter, Firebase, GraphQL",https://example.com/flutter-portfolio
6,"WordPress, PHP, MySQL",https://example.com/wordpress-portfolio
7,"Magento, PHP, MySQL",https://example.com/magento-portfolio
8,"React Native, Node.js, MongoDB",https://example.com/react-native-portfolio
9,"iOS, Swift, Core Data",https://example.com/ios-portfolio


In [47]:
import uuid
import chromadb

client = chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name="portfolio")

if not collection.count():
    for _, row in df.iterrows():
        collection.add(documents=row["Techstack"],
                       metadatas={"links": row["Links"]},
                       ids=[str(uuid.uuid4())])

In [50]:
links = collection.query(
    query_texts = [
        "Experience in Python",
        "Experience in React"
    ],
    n_results = 2
)
print(links)

{'ids': [['619cecca-f971-4038-a7da-856fedf5c495', '94475dcf-9b86-4a37-8ab7-2b015a3edbf2'], ['173537d1-e73f-4c33-a3c0-df9fe2b995d3', 'a34f3e8c-8e1c-45c9-941f-26cc184e625d']], 'embeddings': None, 'documents': [['Machine Learning, Python, TensorFlow', 'Python, Django, MySQL'], ['React, Node.js, MongoDB', 'React Native, Node.js, MongoDB']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'links': 'https://example.com/ml-python-portfolio'}, {'links': 'https://example.com/python-portfolio'}], [{'links': 'https://example.com/react-portfolio'}, {'links': 'https://example.com/react-native-portfolio'}]], 'distances': [[0.9968848824501038, 1.0578700304031372], [0.96307373046875, 1.0685747861862183]]}


In [53]:
job = json_res
job[0]

{'role': 'Retail Associate, PT - Nike Las Vegas South',
 'experience': '',
 'skills': '',
 'description': ''}

In [54]:
prompt_email = PromptTemplate.from_template(
        """
        ### JOB DESCRIPTION:
        {job_description}
        
        ### INSTRUCTION:
        You are Mohan, a business development executive at AtliQ. AtliQ is an AI & Software Consulting company dedicated to facilitating
        the seamless integration of business processes through automated tools. 
        Over our experience, we have empowered numerous enterprises with tailored solutions, fostering scalability, 
        process optimization, cost reduction, and heightened overall efficiency. 
        Your job is to write a cold email to the client regarding the job mentioned above describing the capability of AtliQ 
        in fulfilling their needs.
        Also add the most relevant ones from the following links to showcase Atliq's portfolio: {link_list}
        Remember you are Mohan, BDE at AtliQ. 
        Do not provide a preamble.
        ### EMAIL (NO PREAMBLE):
        
        """
        )

chain_email = prompt_email | llm
res = chain_email.invoke({"job_description": str(job), "link_list": links})
print(res.content)

Subject: Accelerate Nike’s Retail & Product Innovation with AI‑Powered Solutions  

Hi [Recipient Name],

I’m Mohan, Business Development Executive at AtliQ, an AI‑ and software‑consulting firm that partners with leading brands to transform retail operations, product development, and customer experiences through intelligent automation.

I noticed Nike is hiring across several key areas—Retail Associates, Visual Merchandising & Styling, Stockroom Management, Software Engineering, and Footwear Product Line Management. At AtliQ we’ve helped global retailers and apparel manufacturers solve exactly these challenges:

**AI‑Driven Inventory & Stockroom Optimization**  
- Predictive demand forecasting and automated replenishment using machine‑learning models (Python, TensorFlow).  
- Real‑time stockroom dashboards that integrate with existing POS systems, reducing out‑of‑stock incidents by up to 30%.

**Computer‑Vision Visual Merchandising**  
- Automated planogram compliance and style‑trend d